In [1]:
# Import relevant packages
import gamspy as gp
# from gamspy import Container
# from gams import transfer
# import gams.transfer as gt
# import gdxpds
import numpy as np
import pandas as pd
from pathlib import Path
import re
import os

In [2]:
os.getcwd()

'/Users/ahanamukherjee/seer/seer/gams2pypsa'

### Data Pre-Processing & Exploration

In [2]:
# Define a function to load all .gdx files 
def load_gdx_files_simple(data_dir): 

    # Initialize an empty dictionary to store the data 
    all_data = {}
    for country_dir in Path(data_dir).iterdir(): 
        gdx_file = country_dir / 'input.gdx' 
        if gdx_file.exists():
            m = gp.Container() 
            m.read(str(gdx_file)) 
            # Convert all symbols into dataframes 
            country_data = {}
            for symbol_name in m.listSymbols(): 
                country_data[symbol_name] = pd.DataFrame(m[symbol_name].records) 

            all_data[country_dir.name] = country_data 
    
    return all_data

In [3]:
# Load all of the gdx files into a dictionary
all_data_simple = load_gdx_files_simple("data/raw")

In [4]:
# Check if all of the symbol names are the same across all countries
country_symbols = {country: set(symbols.keys()) for country, symbols in all_data_simple.items()}
symbol_check = all(s == list(country_symbols.values())[0] for s in country_symbols.values())
print(f'Does all country data use the same set of symbols? {symbol_check}')
print("")

# Find the "common" symbols across all countries
common_symbols = set.intersection(*country_symbols.values())
print("Common symbols in all countries:")
print(common_symbols)
print("")

# Find country-specific differences
for country, symbols in country_symbols.items():
    unique_to_country = symbols - common_symbols
    if unique_to_country:
        print(f"Symbols unique to {country}: {unique_to_country}")

Does all country data use the same set of symbols? False

Common symbols in all countries:
{'pStorDataExcel', 'pVREgenProfile', 'pTradePrice', 'pAvailability', 'pHours', 'pDemandForecast', 'pEmissionsTotal', 'pCSPData', 'pCarbonPrice', 'pEnergyEfficiencyFactor', 'pTransferLimit', 'ftfindex', 'pDemandData', 'pNewTransmission', 'pDemandProfile', 'pFuelPrice', 'pVREProfile', 'y', 'pMaxFuellimit'}

Symbols unique to Eswatini: {'pMaxExchangeShareTotal', 'pAvailabilityH2', 'pLossFactor', 'pMaxExchangeShare', 'pSpinningReserveReqCountry', 'peak', 'pExternalH2', 'pFuelTypeCarbonContent', 'pH2DataExcel', 'pExtTransferLimit', 'pEmissionsCountry', 'zcmapExcel', 'MapGG', 'pPlanningReserveMargin', 'sTopology', 'pSpinningReserveReqSystem', 'pFuelData', 'pScalars', 'zext', 'Relevant', 'pTechDataExcel', 'hh', 'pCapexTrajectory', 'pZoneIndex', 'pCapexTrajectoryH2', 'pGenDataExcel'}
Symbols unique to Tanzania: {'pMinCap', 'pAvailabilityH2', 'pLossFactor', 'pMaxExchangeShare', 'pSpinningReserveReqCountry

In [5]:
for symbol in common_symbols:
    print(f"\nChecking symbol: {symbol}")
    
    base_country = list(all_data_simple.keys())[0]
    base_cols = set(all_data_simple[base_country][symbol].columns)
    
    for country in all_data_simple:
        cols = set(all_data_simple[country][symbol].columns)
        
        if cols != base_cols:
            print(f"  Differences in {country}:")
            print(f"    Missing: {base_cols - cols}")
            print(f"    Extra:   {cols - base_cols}")


Checking symbol: pEmissionsTotal
  Differences in EAPP:
    Missing: set()
    Extra:   {'value', 'y'}
  Differences in Malawi:
    Missing: set()
    Extra:   {'value', 'uni'}
  Differences in WAPP:
    Missing: set()
    Extra:   {'value', 'uni'}
  Differences in Guinea:
    Missing: set()
    Extra:   {'value', 'uni'}
  Differences in Rwanda:
    Missing: set()
    Extra:   {'value', 'y'}
  Differences in Turkiye:
    Missing: set()
    Extra:   {'value', 'uni'}

Checking symbol: pCSPData
  Differences in Tanzania:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}
  Differences in Malawi:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}
  Differences in WAPP:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}
  Differences in Rwanda:
    Missing: set()
    Extra:   {'g', 'value', 'csphrd', 'shdr'}
  Differences in Turkiye:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}

Checking symbol: pVREProfile
  Dif

In [5]:
# Obtain all of the symbols used across dataset
all_symbols = list(set().union(*country_symbols.values()))
all_symbols.sort()

# Convert to a Pandas series and output to a CSV
all_symbols = pd.Series(all_symbols)
all_symbols.to_csv('data/processed/all_symbols.csv', index=False)

### Input File Processing

In [3]:
# Define a function to build the symbol mapping schema
def build_symbol_registry(mapping_path):

    # Read in the CSV file for symbol mapping
    mapping_df = pd.read_csv(mapping_path)

    # Eliminate any whitespaces in names
    for col in mapping_df.select_dtypes(include='object'):
        mapping_df[col] = mapping_df[col].str.strip()

    # Extract the column names for each symbol
    col_names = [c for c in mapping_df.columns if c.startswith('col_')]

    # Initilize empty dictionary to store symbol mapping
    symbol_registry = {}

    # Build the dictionary
    for _, row in mapping_df.iterrows():

        # Define the raw and standardized symbol name
        resolved = row['canonical']
        raw_name = row['raw']

        # Define the cols
        cols = [row[c] for c in col_names if pd.notna(row[c]) and row[c] != ""]

        # Initialize a new canonical symbol if it doesn't exist in dictionary
        if resolved not in symbol_registry:
            symbol_registry[resolved] = {
                'raw_names': [],
                'columns': cols
            }

        # Append raw name aliases to the canonical symbol
        symbol_registry[resolved]['raw_names'].append(raw_name)

    return symbol_registry


# Define a function to look up the raw symbol name in the symbol registry
def symbol_lookup(raw_symbol, registry):

    for resolved_name, dict_items in registry.items():
        if raw_symbol in dict_items['raw_names']:
            return resolved_name
        
    return None

In [ ]:
# # Define a function infer column names for symbol data
# def infer_names(col_series):
#     """Attempt to infer standardized column name from column value types"""

#     # Extract a sample of data from the column for text analysis
#     sample = col_series.dropna().astype(str).head(50)

#     if sample.empty:
#         return None
    
#     # Hour (t01, t1, t24, etc)
#     if sample.str.match(r"^t\d+$", case=False).all():
#         return "hour"
    
#     # Year detection (4-digit years)
#     if sample.str.match(r"^\d{4}$").all():
#         return "year"

#     # Season detection (dry/wet, Q1, m1, etc.)
#     if sample.str.match(r"^(dry|wet|Q\d+|m\d+)$", case=False).all():
#         return "season"

#     # Data detecton (d1, d2, d3)
#     if sample.str.match(r"^d\d+$", case=False).all():
#         return "day"

#     # Day detection (small integers)
#     if sample.str.match(r"^\d+$").all():
#         max_val = sample.astype(int).max()
#         if max_val <= 31:
#             return "day"
    
#     # Parameter column detection
#     # Heuristic: mostly alphabetic labels
#     if sample.str.match(r"[A-Za-z]+[A-Za-z0-9]"):
#         return "parameters"

#     return None

In [ ]:
def symbol_sync(df, symbol_obj, resolved_name, registry, strict=False):
    """
    Map symbol column names to defined naming schema.
    Align dataframe columns to canonical maximum schema using
    ordered subsequence + left-right anchoring logic.

    Special handling:
        - element_text is protected and mapped explicitly
        - No other column may map to element_text
    """

    if resolved_name not in registry:
        return df

    # Obtain the maximum expected number of columns for the symbol
    expected_cols = registry[resolved_name].get("columns", [])
    if not expected_cols:
        return df

    # If symbol does not have any columns or symbol df is empty, return the original df
    if df.empty or len(df.columns) == 0:
        return pd.DataFrame(columns=expected_cols)

    # Create a copy of the symbol dataframe for column re-naming and processing
    df = df.copy()

    # Check that "value" column exists or rename last column to value
    if "value" not in df.columns:
        df = df.rename(columns={df.columns[-1]: "value"})

    # Extract all expected column names from schema excluding "value" column
    expected_dims = [c for c in expected_cols if c != "value"]

    # Extract all actual column names from df excluding "value" column
    actual_dim_cols = [c for c in df.columns if c != "value"]

    # Explicitly map element_text column if present in dataframe
    rename_map = {}

    # Remove element_text from canonical columns for dataframe
    if "element_text" in expected_dims and "element_text" in df.columns:
        rename_map["element_text"] = "element_text"
        actual_dim_cols.remove("element_text")

    # Determine the remaining dimensions that need to be mapped
    remaining_dims = [
        d for d in expected_dims if d != "element_text"
    ]

    # Use a left-to-right anchoring scheme for renaming columns
    # Determine the total number of columns that need to be mapped
    n = len(actual_dim_cols)
    # n = len(dims_for_alignment)

    def left_to_right_mapping(expected_dims, n):
        if n == 0:
            return []

        if n >= len(expected_dims):
            return expected_dims[:n]

        if n == 1:
            return [expected_dims[0]]

        left_anchor = expected_dims[0]
        right_anchor = expected_dims[-1]

        remaining = n - 2
        middle = expected_dims[1:-1]

        return (
            [left_anchor] +
            middle[:remaining] +
            [right_anchor]
        )

    # Apply mapping for remaining columns based on column order defined in schema
    mapped_dims = left_to_right_mapping(remaining_dims, n)

    # Assign renamed dimensions
    for i in range(len(mapped_dims)):
        rename_map[actual_dim_cols[i]] = mapped_dims[i]
    # Rename columns according to mapped dimensions
    df = df.rename(columns=rename_map)

    # Add extra columns that are not in the actual df but in expected column schema as NaNs
    for col in expected_dims:
        if col not in df.columns:
            df[col] = np.nan

    # If "value" column doens't exist in the dataframe, add column with NaNs
    if "value" not in df.columns:
        df["value"] = np.nan

    # Reorder columns to match the expected column schema
    df = df[expected_cols]

    # Raise ValueError if mismatch across expected schema and actual columns
    if strict:
        if list(df.columns) != expected_cols:
            raise ValueError(
                f"{resolved_name}: schema mismatch after alignment"
            )

    return df

In [8]:
# Define a function to load all .gdx files 
def load_gdx_files(data_dir, mapping_path, strict): 

    # Build the symbol mapping registry
    registry = build_symbol_registry(mapping_path)

    # Initialize an empty dictionary to store the data 
    all_data = {}
    for country_dir in Path(data_dir).iterdir(): 
        gdx_file = country_dir / 'input.gdx' 
        if gdx_file.exists():

            print(f'Processing {country_dir.name}')

            m = gp.Container() 
            m.read(str(gdx_file)) 

            # Convert all symbols into dataframes 
            country_data = {}

            for raw_symbol in m.listSymbols(): 

                # Find the corresponding resolved symbol name in the symbol registry
                resolved_symbol = symbol_lookup(raw_symbol, registry)

                if resolved_symbol is None:
                    continue
                
                # Convert the symbol records into a dataframe
                symbol_obj = m[raw_symbol]
                df = pd.DataFrame(symbol_obj.records)

                # Rename columns of the symbol data
                df = symbol_sync(df, symbol_obj, resolved_symbol, registry, strict)
                
                # Append cleaned df to country
                country_data[resolved_symbol] = df

            all_data[country_dir.name] = country_data 
    
    return all_data

In [9]:
# Load all of the gdx files into a dictionary
all_data = load_gdx_files(data_dir="data/raw", 
                          mapping_path="data/references/symbol_mapping.csv", 
                          strict=True)

Processing Eswatini
Processing Tanzania
Processing EAPP
Processing Malawi
Processing WAPP
Processing Serbia
Processing Timor Leste
Processing Gambia
Processing Guinea
Processing CAPP
Processing Rwanda
Processing Turkiye


In [14]:
# Define a function to write out all model data into csvs
def output_to_csv(all_data, output_dir):

    # Create the output directory if it does not exist
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Loop through each country/region and generate csv for each dataframe
    for region_name, region_data in all_data.items():

        # Create region-specific directory
        region_dir = output_dir / region_name
        region_dir.mkdir(parents=True, exist_ok=True)

        # Loop through the symbol dataframes for each region
        for symbol_name, df in region_data.items():

            # Skip writing empty dataframes
            if df is None or df.empty:
                continue

            # Define the file path and write out the dataframe to a csv
            file_path = region_dir / f'{symbol_name}.csv'
            df.to_csv(file_path, index=False)
        
        # Print logging message
        print(f"Finished writing all symbols for {region_name}")

    return

In [15]:
# Write out symbol dataframes for each region
output_to_csv(all_data, output_dir='data/processed')

Finished writing all symbols for Eswatini
Finished writing all symbols for Tanzania
Finished writing all symbols for EAPP
Finished writing all symbols for Malawi
Finished writing all symbols for WAPP
Finished writing all symbols for Serbia
Finished writing all symbols for Timor Leste
Finished writing all symbols for Gambia
Finished writing all symbols for Guinea
Finished writing all symbols for CAPP
Finished writing all symbols for Rwanda
Finished writing all symbols for Turkiye
